In [2]:
# ==============================
# Data Handling
# ==============================
import numpy as np
import pandas as pd

# ==============================
# Visualization
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# TensorFlow / Keras
# ==============================
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout,
    Bidirectional,
    Flatten
)

# ==============================
# Preprocessing
# ==============================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ==============================
# Evaluation
# ==============================
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [27]:
df=pd.read_csv("quotes.csv")
df = df.dropna(subset=["quote"])
df.head()

,index,quote,author,tags,likes
0,0,Be yourself; everyone else is already taken.,Oscar Wilde,attributed-no-source;be-yourself;honesty;inspi...,149270
1,1,You've gotta dance like there's nobody watching,William W. Purkey,dance;heaven;hurt;inspirational;life;love;sing,118888
2,2,Be the change that you wish to see in the world.,Mahatma Gandhi,action;change;inspirational;philosophy;wish,106749
3,3,No one can make you feel inferior without your...,"Eleanor Roosevelt,",confidence;inspirational;wisdom,85854
4,4,Live as if you were to die tomorrow. Learn as ...,Mahatma Gandhi,carpe-diem;education;inspirational;learning,73033


In [28]:
quote=df["quote"]
quote

0            Be yourself; everyone else is already taken.
1         You've gotta dance like there's nobody watching
2        Be the change that you wish to see in the world.
3       No one can make you feel inferior without your...
4       Live as if you were to die tomorrow. Learn as ...
                              ...                        
2996    And this is for Colored girls who have conside...
2997    After all, when you come right down to it, how...
2998    Aku telah mengidap sakit gila nomor enam belas...
2999    The moon is the reflection of your heart and m...
3000    But in the end it's only a passing thing, this...
Name: quote, Length: 2996, dtype: str

In [29]:
import string



quote = (
   quote
    .str.lower()
    .str.translate(str.maketrans("", "", string.punctuation))
    .str.strip()
)

In [30]:
quote.isnull().sum()

np.int64(0)

In [31]:
vocab_size=10000
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quote)
sequences = tokenizer.texts_to_sequences(quote)

In [32]:
sequences


[[13, 60, 217, 168, 5, 518, 719],
 [261, 846, 396, 55, 284, 497, 1133],
 [13, 1, 102, 10, 3, 327, 2, 80, 9, 1, 42],
 [35, 31, 23, 52, 3, 128, 3515, 132, 12, 2393],
 [76, 29, 22, 3, 93, 2, 198, 285, 136, 29, 22, 3, 93, 2, 76, 373],
 [359,
  150,
  921,
  73,
  359,
  45,
  191,
  23,
  25,
  10,
  313,
  150,
  921,
  73,
  313,
  45,
  39,
  23,
  25,
  10],
 [132, 439, 21, 117, 13, 7, 537],
 [15, 314, 1, 39, 15, 79, 15, 720],
 [1837, 5, 222, 1134, 5, 1838, 4, 50, 113, 2, 13, 1537, 1839, 54, 1537, 1538],
 [34,
  16,
  45,
  196,
  478,
  2,
  76,
  12,
  21,
  31,
  5,
  29,
  246,
  97,
  5,
  7,
  922,
  1,
  94,
  5,
  29,
  246,
  104,
  5,
  7,
  922],
 [15, 16, 24, 9, 1, 3516, 20, 118, 6, 58, 16, 315, 44, 1, 328],
 [1135,
  1539,
  16,
  56,
  54,
  169,
  14,
  59,
  33,
  193,
  58,
  10,
  1281,
  721,
  20,
  59,
  33,
  193,
  58,
  10,
  1281,
  23,
  13,
  1840],
 [498,
  5,
  456,
  285,
  5,
  7,
  1020,
  170,
  5,
  7,
  291,
  6,
  98,
  95,
  5,
  201,
  15,
  457,
 

In [33]:
df["quote_tokens"] = tokenizer.texts_to_sequences(quote)

In [34]:
df

,index,quote,author,tags,likes,quote_tokens
0,0,Be yourself; everyone else is already taken.,Oscar Wilde,attributed-no-source;be-yourself;honesty;inspi...,149270,"[13, 60, 217, 168, 5, 518, 719]"
1,1,You've gotta dance like there's nobody watching,William W. Purkey,dance;heaven;hurt;inspirational;life;love;sing,118888,"[261, 846, 396, 55, 284, 497, 1133]"
2,2,Be the change that you wish to see in the world.,Mahatma Gandhi,action;change;inspirational;philosophy;wish,106749,"[13, 1, 102, 10, 3, 327, 2, 80, 9, 1, 42]"
3,3,No one can make you feel inferior without your...,"Eleanor Roosevelt,",confidence;inspirational;wisdom,85854,"[35, 31, 23, 52, 3, 128, 3515, 132, 12, 2393]"
4,4,Live as if you were to die tomorrow. Learn as ...,Mahatma Gandhi,carpe-diem;education;inspirational;learning,73033,"[76, 29, 22, 3, 93, 2, 198, 285, 136, 29, 22, ..."
...,...,...,...,...,...,...
2996,2996,And this is for Colored girls who have conside...,"Ntozake Shange,",inspirational,63,"[4, 41, 5, 17, 3489, 1287, 28, 18, 1255, 1491,..."
2997,2997,"After all, when you come right down to it, how...",Russell Hoban,inspirational,61,"[205, 24, 30, 3, 121, 92, 163, 2, 8, 64, 175, ..."
2998,2998,Aku telah mengidap sakit gila nomor enam belas...,"Andrea Hirata,",humor;inspirational;irony,61,"[2892, 7732, 7733, 7734, 7735, 7736, 7737, 773..."
2999,2999,The moon is the reflection of your heart and m...,Debasish Mridha,education;happiness;heart;hope;inspirational;i...,58,"[1, 1185, 5, 1, 1085, 6, 12, 87, 4, 7751, 5, 1..."


In [68]:
X = []
y = []
y1=[]
for sequence in df["quote_tokens"]:

    for i in range(1, len(sequence)):

        X.append(sequence[:i])
        y.append(sequence[i])
        y1.append(sequence[i])

In [36]:
print(X[:5])
print(y[:5])

[[13], [13, 60], [13, 60, 217], [13, 60, 217, 168], [13, 60, 217, 168, 5]]
[60, 217, 168, 5, 518]


In [69]:
y1

[60,
 217,
 168,
 5,
 518,
 719,
 846,
 396,
 55,
 284,
 497,
 1133,
 1,
 102,
 10,
 3,
 327,
 2,
 80,
 9,
 1,
 42,
 31,
 23,
 52,
 3,
 128,
 3515,
 132,
 12,
 2393,
 29,
 22,
 3,
 93,
 2,
 198,
 285,
 136,
 29,
 22,
 3,
 93,
 2,
 76,
 373,
 150,
 921,
 73,
 359,
 45,
 191,
 23,
 25,
 10,
 313,
 150,
 921,
 73,
 313,
 45,
 39,
 23,
 25,
 10,
 439,
 21,
 117,
 13,
 7,
 537,
 314,
 1,
 39,
 15,
 79,
 15,
 720,
 5,
 222,
 1134,
 5,
 1838,
 4,
 50,
 113,
 2,
 13,
 1537,
 1839,
 54,
 1537,
 1538,
 16,
 45,
 196,
 478,
 2,
 76,
 12,
 21,
 31,
 5,
 29,
 246,
 97,
 5,
 7,
 922,
 1,
 94,
 5,
 29,
 246,
 104,
 5,
 7,
 922,
 16,
 24,
 9,
 1,
 3516,
 20,
 118,
 6,
 58,
 16,
 315,
 44,
 1,
 328,
 1539,
 16,
 56,
 54,
 169,
 14,
 59,
 33,
 193,
 58,
 10,
 1281,
 721,
 20,
 59,
 33,
 193,
 58,
 10,
 1281,
 23,
 13,
 1840,
 5,
 456,
 285,
 5,
 7,
 1020,
 170,
 5,
 7,
 291,
 6,
 98,
 95,
 5,
 201,
 15,
 457,
 8,
 1,
 346,
 18,
 14,
 1282,
 316,
 70,
 334,
 3517,
 478,
 10,
 458,
 149,
 1021,
 6,
 39,
 

In [41]:
X

[[13],
 [13, 60],
 [13, 60, 217],
 [13, 60, 217, 168],
 [13, 60, 217, 168, 5],
 [13, 60, 217, 168, 5, 518],
 [261],
 [261, 846],
 [261, 846, 396],
 [261, 846, 396, 55],
 [261, 846, 396, 55, 284],
 [261, 846, 396, 55, 284, 497],
 [13],
 [13, 1],
 [13, 1, 102],
 [13, 1, 102, 10],
 [13, 1, 102, 10, 3],
 [13, 1, 102, 10, 3, 327],
 [13, 1, 102, 10, 3, 327, 2],
 [13, 1, 102, 10, 3, 327, 2, 80],
 [13, 1, 102, 10, 3, 327, 2, 80, 9],
 [13, 1, 102, 10, 3, 327, 2, 80, 9, 1],
 [35],
 [35, 31],
 [35, 31, 23],
 [35, 31, 23, 52],
 [35, 31, 23, 52, 3],
 [35, 31, 23, 52, 3, 128],
 [35, 31, 23, 52, 3, 128, 3515],
 [35, 31, 23, 52, 3, 128, 3515, 132],
 [35, 31, 23, 52, 3, 128, 3515, 132, 12],
 [76],
 [76, 29],
 [76, 29, 22],
 [76, 29, 22, 3],
 [76, 29, 22, 3, 93],
 [76, 29, 22, 3, 93, 2],
 [76, 29, 22, 3, 93, 2, 198],
 [76, 29, 22, 3, 93, 2, 198, 285],
 [76, 29, 22, 3, 93, 2, 198, 285, 136],
 [76, 29, 22, 3, 93, 2, 198, 285, 136, 29],
 [76, 29, 22, 3, 93, 2, 198, 285, 136, 29, 22],
 [76, 29, 22, 3, 93, 2

In [43]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X = pad_sequences(X, padding="pre")

In [44]:
X

array([[  0,   0,   0, ...,   0,   0,  13],
       [  0,   0,   0, ...,   0,  13,  60],
       [  0,   0,   0, ...,  13,  60, 217],
       ...,
       [  0,   0,   0, ...,  41, 974,  91],
       [  0,   0,   0, ..., 974,  91, 359],
       [  0,   0,   0, ...,  91, 359, 106]],
      shape=(79812, 287), dtype=int32)

In [45]:
y=np.array(y)

In [46]:
y

array([ 60, 217, 168, ..., 359, 106, 623], shape=(79812,))

In [47]:
X.shape

(79812, 287)

In [48]:
y.shape

(79812,)

In [57]:
vocab_size = len(tokenizer.word_index)

print("Maximum vocabulary:", vocab_size)

Maximum vocabulary: 7751


In [58]:
print(tokenizer.word_index)

{'the': 1, 'to': 2, 'you': 3, 'and': 4, 'is': 5, 'of': 6, 'a': 7, 'it': 8, 'in': 9, 'that': 10, 'i': 11, 'your': 12, 'be': 13, 'not': 14, 'we': 15, 'are': 16, 'for': 17, 'have': 18, 'what': 19, 'but': 20, 'life': 21, 'if': 22, 'can': 23, 'all': 24, 'do': 25, 'with': 26, 'will': 27, 'who': 28, 'as': 29, 'when': 30, 'one': 31, 'or': 32, 'they': 33, 'there': 34, 'no': 35, 'people': 36, 'on': 37, 'my': 38, 'love': 39, 'our': 40, 'this': 41, 'world': 42, 'never': 43, 'at': 44, 'only': 45, 'me': 46, 'he': 47, 'things': 48, 'dont': 49, 'its': 50, 'so': 51, 'make': 52, 'by': 53, 'than': 54, 'like': 55, 'more': 56, 'them': 57, 'us': 58, 'because': 59, 'yourself': 60, 'know': 61, 'up': 62, 'was': 63, 'how': 64, 'from': 65, 'time': 66, 'always': 67, 'about': 68, 'want': 69, 'just': 70, 'way': 71, 'their': 72, 'out': 73, 'something': 74, 'own': 75, 'live': 76, 'get': 77, 'an': 78, 'think': 79, 'see': 80, 'go': 81, 'let': 82, 'has': 83, 'am': 84, 'every': 85, 'thing': 86, 'heart': 87, 'being': 88, 

In [73]:
from tensorflow.keras.utils import to_categorical
y1=np.array(y1)
vocab_size = max(y1.max(), max(tokenizer.word_index.values())) + 1

print("Vocabulary size:", vocab_size)
print("Maximum y1 value:", y1.max())
y_one_hot = to_categorical(
    y1,
    num_classes=vocab_size
)

print("y_one_hot shape:", y_one_hot.shape)

Vocabulary size: 7752
Maximum y1 value: 7751
y_one_hot shape: (79812, 7752)


In [74]:
y_one_hot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(79812, 7752))

In [78]:
X = pad_sequences(
    X,
   
    padding="post",
)


In [79]:
X

array([[ 13,   0,   0, ...,   0,   0,   0],
       [ 13,  60,   0, ...,   0,   0,   0],
       [ 13,  60, 217, ...,   0,   0,   0],
       ...,
       [ 20,   9,   1, ...,   0,   0,   0],
       [ 20,   9,   1, ...,   0,   0,   0],
       [ 20,   9,   1, ...,   0,   0,   0]],
      shape=(79812, 287), dtype=int32)

In [81]:
vocab_size = int(max(y1.max(), max(tokenizer.word_index.values())) + 1)

print(vocab_size)
print(type(vocab_size))

7752
<class 'int'>


In [82]:
model = Sequential([
    Input(shape=(X.shape[1],)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        mask_zero=True
    ),

    SimpleRNN(128),

    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)